# 00 — ZTBus quickstart

Interactive walkthrough of the cleaning → features → forward-sim pipeline.
Run this AFTER `ztbus ingest` has produced interim Parquet files.

Prerequisites:
```bash
make install
uv run ztbus ingest --raw-dir <path-to-csvs>
```


In [ ]:
import polars as pl
from pathlib import Path

from ztbus.cleaning import CleaningConfig, clean_mission
from ztbus.cleaning.grade import derive_grade
from ztbus.features import add_kinematics, add_mass, add_energy
from ztbus.routes import detect_depot_phases
from ztbus.qc import run_gates
from ztbus.physics import simulate_powertrain, PowertrainParameters, PhysicalConstants

INTERIM = Path("../data/interim")
files = sorted(INTERIM.rglob("*.parquet"))
print(f"{len(files)} interim mission files available")

## 1. Pick one mission and run the full pipeline

In [ ]:
f = files[0]
bus = int(next(p.split("=")[1] for p in f.parts if p.startswith("bus=")))
df_raw = pl.read_parquet(f)
print(f"Loaded {f.stem}: {df_raw.height} rows, {df_raw.width} cols")

cfg = CleaningConfig.from_yaml("../configs/cleaning/v1.yaml")
df, qc = clean_mission(df_raw, mission_id=f.stem, bus=bus, cfg=cfg)
df = add_kinematics(df)
df = derive_grade(df, cfg.grade)
df = add_mass(df)
df = add_energy(df)
df, depot = detect_depot_phases(df)
print(qc.flag_counts)
print(f"Depot trim: start={depot.n_rows_start_depot}, end={depot.n_rows_end_depot}")

## 2. Run QC gates

In [ ]:
for g in run_gates(df):
    print(f"{'PASS' if g.passed else 'FAIL':6} {g.name:32} {g.detail}")

## 3. Forward-simulate with the prior parameters

In [ ]:
ix = ~df['in_depot']
non_depot = df.filter(ix)

sim = simulate_powertrain(
    time_s=non_depot['time_unix'].to_numpy().astype(float),
    speed_mps=non_depot['speed_smoothed_mps'].to_numpy().astype(float),
    acceleration_mps2=non_depot['acceleration_mps2'].to_numpy().astype(float),
    mass_kg=non_depot['mass_kg'].to_numpy().astype(float),
    grade=non_depot['grade'].to_numpy().astype(float),
    temperature_K=non_depot['temperature_ambient'].to_numpy().astype(float),
)

import numpy as np
P_meas = non_depot['electric_powerDemand'].to_numpy().astype(float)
P_sim = sim.power_total_W
rmse_kW = np.sqrt(np.nanmean((P_sim - P_meas)**2)) / 1000
print(f"RMSE (priors): {rmse_kW:.1f} kW")
print(f"Mean meas: {np.nanmean(P_meas)/1000:.1f} kW")
print(f"Mean sim:  {np.nanmean(P_sim)/1000:.1f} kW")